# Tutorial 02: Applied Microdata Workflow & Project STAR Case Study

This notebook shows how to:
1. Build an `AppliedDataset` from individual/household microdata grouped by aggregate experimental units (classrooms, state-years, villages) using `quantile_matrix`, `weighted_quantiles`, and `ecdf_moderator`.
2. Fit WCF (`run_wcf`) and a permutation placebo test (`run_placebo`).
3. Load and inspect the audited **Project STAR** classroom-randomization analysis ($n = 1{,}333$ classroom-grades) from Section 4 of the paper.

In [ ]:
import os
import sys
from pathlib import Path

if not Path("src/wasserstein_causal_forests").exists() and Path("../src/wasserstein_causal_forests").exists():
    os.chdir("..")
if Path("src").exists() and str(Path("src").resolve()) not in sys.path:
    sys.path.insert(0, str(Path("src").resolve()))

In [ ]:
import json
from pathlib import Path
import numpy as np

from wasserstein_causal_forests import (
    AppliedDataset,
    ecdf_moderator,
    midpoint_levels,
    quantile_matrix,
    run_placebo,
    run_wcf,
    weighted_quantiles,
)

rng = np.random.default_rng(123)
n_units = 160
individuals_per_unit = rng.integers(15, 35, size=n_units)
poverty_share = rng.uniform(0.10, 0.90, size=n_units)
urban = rng.binomial(1, 0.45, size=n_units).astype(float)
teacher_exp = rng.gamma(shape=3.0, scale=3.0, size=n_units)
A = rng.binomial(1, 0.45, size=n_units).astype(np.int64)

unit_ids_list, outcomes_list, weights_list = [], [], []
for unit_id in range(n_units):
    m_i = int(individuals_per_unit[unit_id])
    unit_shift = (
        -0.40 * poverty_share[unit_id]
        + 0.02 * teacher_exp[unit_id]
        + A[unit_id] * (0.18 + 0.15 * poverty_share[unit_id])
        + rng.normal(0.0, 0.15)
    )
    unit_ids_list.append(np.full(m_i, unit_id, dtype=int))
    outcomes_list.append(unit_shift + rng.normal(0.0, 1.0, size=m_i))
    weights_list.append(rng.uniform(0.8, 1.2, size=m_i))

all_unit_ids = np.concatenate(unit_ids_list)
all_outcomes = np.concatenate(outcomes_list)
all_weights = np.concatenate(weights_list)

u = midpoint_levels(25)
_, Q = quantile_matrix(all_outcomes, all_weights, all_unit_ids, levels=u)
control_indiv = np.isin(all_unit_ids, np.where(A == 0)[0])
q_star = weighted_quantiles(all_outcomes[control_indiv], all_weights[control_indiv], levels=u)
X = np.column_stack([ecdf_moderator(poverty_share), urban, teacher_exp])

dataset = AppliedDataset(
    study="synthetic_classroom_demo",
    X=X,
    A=A,
    Q=Q,
    moderator_raw=poverty_share,
    q_star=q_star,
    feature_names=["poverty_share_ecdf", "urban", "teacher_exp"],
).validate()

idx_p10 = int(np.argmin(np.abs(u - 0.10)))
idx_p90 = int(np.argmin(np.abs(u - 0.90)))
extra = {
    "p10": lambda q, k=idx_p10: np.asarray(q, dtype=float)[:, k],
    "p90": lambda q, k=idx_p90: np.asarray(q, dtype=float)[:, k],
}

_, results = run_wcf(dataset, random_state=0, n_estimators=50, extra_functionals=extra)
placebo = run_placebo(dataset, seeds=(0,), n_estimators=50, extra_functionals=extra)[0]

for func in ("mean", "sd", "skew", "tail", "reference", "p10", "p90"):
    print(
        f"{func:<12} DR={results['marginal_dr'][func]:+8.3f} "
        f"(SE={results['if_se'][func]:.3f}) | Placebo={placebo['marginal_dr'][func]:+8.3f}"
    )

In [ ]:
star_dir = Path("results/applied_study_exploration/project_star")
star_ds = AppliedDataset.load(star_dir / "data")
star_summary = json.loads((star_dir / "summary.json").read_text(encoding="utf-8"))

print(f"Project STAR: n={star_ds.X.shape[0]} classrooms (treated={int(star_ds.A.sum())}, control={int((star_ds.A == 0).sum())})")
for func, vals in star_summary["primary"]["functionals"].items():
    print(f"  {func:<12}: DR={vals['marginal_mean']:+.3f} (IF SE={vals['if_se_mean']:.3f}), Plug-in={vals['plugin_marginal_mean']:+.3f}")

output_file = "tutorial_02_applied_results.json"
Path(output_file).write_text(json.dumps(results, indent=2), encoding="utf-8")

In [ ]:
try:
    from google.colab import files
    files.download(output_file)
    print("Downloaded:", output_file)
except Exception as e:
    print("(Not on Colab / download skipped):", e)